In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/__results__.html
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/__notebook__.ipynb
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/__output__.json
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/custom.css
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/benchmark.jsonl
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/test.jsonl
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/README.md
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/validation.jsonl
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/capability_examples.csv
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/metadata.json
/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/train.jso

In [2]:
# %% [code]
# =============================================================================
# CELL 1 — ENVIRONMENT / DEPENDENCIES
# =============================================================================

!pip install -q -U "peft>=0.19.0" "datasets>=3.0.0" "transformers>=5.0.0" scikit-learn psutil

import os
import sys
import json
import math
import time
import random
import shutil
import gc
import re
import subprocess
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
import transformers
import datasets
import peft

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model

print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("PEFT         :", peft.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("CUDA version :", torch.version.cuda)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.
ENVIRONMENT
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
Transformers : 5.17.0
Datasets     : 5.0.1
PEFT         : 0.20.0
CUDA         : True
GPU      

In [3]:
# %% [code]
# =============================================================================
# CELL 2 — CONFIGURATION
# =============================================================================

SEED = 20260910
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_MODEL = "google/gemma-3-270m-it"
MODEL_DTYPE = torch.float32

PROJECT_ROOT = Path("/kaggle/working")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
LORA_DIR = MODELS_DIR / "lora"
MERGED_DIR = MODELS_DIR / "merged"
GGUF_DIR = MODELS_DIR / "gguf"
REPORTS_DIR = PROJECT_ROOT / "reports"
EXPORT_DIR = PROJECT_ROOT / "export"

for d in [
    DATA_DIR, PROCESSED_DIR, MODELS_DIR, LORA_DIR, MERGED_DIR,
    GGUF_DIR, REPORTS_DIR, EXPORT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# SME-Ledger V2 Input Directory (Locked directly to generator outputs)
# -------------------------------------------------------------------------
V2_DATASET_ROOT = Path(
    "/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset"
)

def find_v2_dataset():
    if V2_DATASET_ROOT.exists() and (V2_DATASET_ROOT / "train.jsonl").exists():
        return V2_DATASET_ROOT

    # Fallback search if path structure varies in environment
    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not root.exists():
            continue
        for p in root.rglob("train.jsonl"):
            parent = p.parent
            if (
                (parent / "validation.jsonl").exists()
                and (parent / "test.jsonl").exists()
                and (parent / "benchmark.jsonl").exists()
            ):
                candidates.append(parent)

    candidates.sort(key=lambda p: ("sme" not in str(p).lower(), len(str(p))))
    return candidates[0] if candidates else None

V2_ROOT = find_v2_dataset()

if V2_ROOT is None:
    raise FileNotFoundError(
        f"SME-Ledger V2 dataset was not found at {V2_DATASET_ROOT}. "
        "Ensure the SME-Ledger generator output is properly attached."
    )

# Dataset Split Paths
TRAIN_PATH = V2_ROOT / "train.jsonl"
VAL_PATH = V2_ROOT / "validation.jsonl"
TEST_PATH = V2_ROOT / "test.jsonl"
BENCHMARK_PATH = V2_ROOT / "benchmark.jsonl"

# Auxiliary File Paths
ALL_EXAMPLES_PATH = V2_ROOT / "all_examples.jsonl"
TRANSACTIONS_PATH = V2_ROOT / "transactions.csv"
CAPABILITY_PATH = V2_ROOT / "capability_examples.csv"
METADATA_PATH = V2_ROOT / "metadata.json"

# LoRA Configuration
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Training Hyperparameters
LEARNING_RATE = 1e-4
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_LENGTH = 768
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

MAX_NEW_TOKENS = 300

print("Base model :", BASE_MODEL)
print("V2 dataset :", V2_ROOT)
print("Train      :", TRAIN_PATH)
print("Validation :", VAL_PATH)
print("Test       :", TEST_PATH)
print("Benchmark  :", BENCHMARK_PATH)
print("Max length :", MAX_LENGTH)

Base model : google/gemma-3-270m-it
V2 dataset : /kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset
Train      : /kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/train.jsonl
Validation : /kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/validation.jsonl
Test       : /kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/test.jsonl
Benchmark  : /kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset/benchmark.jsonl
Max length : 768


In [4]:
# %% [code]
# =============================================================================
# CELL 3 — MEMORY UTILITIES
# =============================================================================

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_memory():
    if not torch.cuda.is_available():
        return {}
    return {
        "allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 3),
        "reserved_gb": round(torch.cuda.memory_reserved() / 2**30, 3),
    }

def print_memory(label=""):
    print(f"\nMEMORY — {label}")
    print(
        "CPU RSS:",
        round(psutil.Process(os.getpid()).memory_info().rss / 2**30, 3),
        "GB"
    )
    if torch.cuda.is_available():
        mem = gpu_memory()
        print("GPU allocated:", mem["allocated_gb"], "GB")
        print("GPU reserved :", mem["reserved_gb"], "GB")

In [5]:
# %% [code]
# =============================================================================
# CELL 4 — LOAD HUGGING FACE TOKEN FROM KAGGLE SECRETS
# =============================================================================

try:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

    if not HF_TOKEN:
        raise ValueError("HF_TOKEN secret exists but is empty.")

    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✓ HF_TOKEN loaded from Kaggle Secrets")

except Exception as e:
    print("HF_TOKEN was not loaded.")
    print(type(e).__name__, ":", e)
    print("If the Gemma repository is already cached/publicly accessible, this may still work.")

✓ HF_TOKEN loaded from Kaggle Secrets


In [6]:
# %% [code]
# =============================================================================
# CELL 5 — LOAD SME-LEDGER V2 JSONL
# =============================================================================

def load_jsonl_records(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSONL at {path}:{line_no}: {e}") from e
    return records

train_records = load_jsonl_records(TRAIN_PATH)
val_records = load_jsonl_records(VAL_PATH)
test_records = load_jsonl_records(TEST_PATH)
benchmark_records = load_jsonl_records(BENCHMARK_PATH)

print("=" * 80)
print("SME-LEDGER V2 DATASET")
print("=" * 80)
print("Train      :", len(train_records))
print("Validation :", len(val_records))
print("Test       :", len(test_records))
print("Benchmark  :", len(benchmark_records))

sample = train_records[0]
print("\nSample record keys:", list(sample.keys()))
print("Message roles:", [m.get("role") for m in sample.get("messages", [])])
print("User sample:", sample["messages"][0]["content"][:300])
print("Assistant sample:", sample["messages"][1]["content"][:500])


SME-LEDGER V2 DATASET
Train      : 15624
Validation : 1953
Test       : 1954
Benchmark  : 2000

Sample record keys: ['messages']
Message roles: ['user', 'assistant']
User sample: UIPB1SKBO1 Confirmed. You have received Ksh18,810.00 from Victor Kiptoo 0796617790 on 23/4/26 at 03:41. New M-PESA balance is Ksh108,890.68.
Assistant sample: {"transaction_id":"UIPB1SKBO1","date":"2026-04-23","time":"03:41","type":"income","domain":"receive_money","entity":"Victor Kiptoo","amount":18810.0,"balance":108890.68,"fee":0.0,"reference":null}


In [7]:
# %% [code]
# =============================================================================
# CELL 6 — VALIDATE V2 MESSAGE FORMAT
# =============================================================================

def validate_v2_records(records, name):
    errors = []
    for i, record in enumerate(records):
        messages = record.get("messages")
        if not isinstance(messages, list) or len(messages) != 2:
            errors.append((i, "messages must contain exactly user + assistant"))
            continue

        roles = [m.get("role") for m in messages]
        if roles != ["user", "assistant"]:
            errors.append((i, f"unexpected roles: {roles}"))

        for j, msg in enumerate(messages):
            if not isinstance(msg.get("content"), str) or not msg["content"].strip():
                errors.append((i, f"message {j} has empty/non-string content"))

    if errors:
        print(f"{name}: {len(errors)} invalid records")
        for err in errors[:10]:
            print(" ", err)
        raise ValueError(f"{name} failed V2 format validation.")

    print(f"✓ {name}: {len(records):,} valid records")

for records, name in [
    (train_records, "TRAIN"),
    (val_records, "VALIDATION"),
    (test_records, "TEST"),
    (benchmark_records, "BENCHMARK"),
]:
    validate_v2_records(records, name)

# Inspect V2 source-kind distribution when metadata is present in the generator output.
# JSONL deliberately stores only `messages`, so source_kind is not available here.
print("\n✓ V2 JSONL uses the generator's exact two-message chat format.")


✓ TRAIN: 15,624 valid records
✓ VALIDATION: 1,953 valid records
✓ TEST: 1,954 valid records
✓ BENCHMARK: 2,000 valid records

✓ V2 JSONL uses the generator's exact two-message chat format.


In [8]:
# %% [code]
# =============================================================================
# CELL 7 — V2 DATASET STATISTICS / TARGET INSPECTION
# =============================================================================

def parse_json_if_possible(text):
    try:
        return json.loads(text)
    except Exception:
        return None

def dataset_stats(records, name):
    users = [r["messages"][0]["content"] for r in records]
    assistants = [r["messages"][1]["content"] for r in records]

    json_targets = [parse_json_if_possible(x) for x in assistants]
    json_count = sum(x is not None for x in json_targets)

    target_kinds = {"object": 0, "array": 0, "text": 0}
    for parsed in json_targets:
        if isinstance(parsed, dict):
            target_kinds["object"] += 1
        elif isinstance(parsed, list):
            target_kinds["array"] += 1
        else:
            target_kinds["text"] += 1

    print(f"\n{name}")
    print("-" * 60)
    print("Records           :", len(records))
    print("JSON-valid targets:", json_count)
    print("Objects           :", target_kinds["object"])
    print("Arrays            :", target_kinds["array"])
    print("Plain-text targets:", target_kinds["text"])
    print("Avg user chars    :", round(np.mean([len(x) for x in users]), 1))
    print("Avg target chars  :", round(np.mean([len(x) for x in assistants]), 1))

for records, name in [
    (train_records, "TRAIN"),
    (val_records, "VALIDATION"),
    (test_records, "TEST"),
    (benchmark_records, "BENCHMARK"),
]:
    dataset_stats(records, name)

print("\nExample target:")
print(train_records[0]["messages"][1]["content"])



TRAIN
------------------------------------------------------------
Records           : 15624
JSON-valid targets: 15598
Objects           : 15202
Arrays            : 396
Plain-text targets: 26
Avg user chars    : 163.9
Avg target chars  : 212.8

VALIDATION
------------------------------------------------------------
Records           : 1953
JSON-valid targets: 1952
Objects           : 1891
Arrays            : 61
Plain-text targets: 1
Avg user chars    : 166.4
Avg target chars  : 216.4

TEST
------------------------------------------------------------
Records           : 1954
JSON-valid targets: 1950
Objects           : 1907
Arrays            : 43
Plain-text targets: 4
Avg user chars    : 163.8
Avg target chars  : 211.0

BENCHMARK
------------------------------------------------------------
Records           : 2000
JSON-valid targets: 2000
Objects           : 2000
Arrays            : 0
Plain-text targets: 0
Avg user chars    : 138.7
Avg target chars  : 204.6

Example target:
{"transacti

In [9]:
# %% [code]
# =============================================================================
# CELL 8 — PRESERVE THE GENERATOR SPLITS
# =============================================================================
#
# IMPORTANT:
# Do NOT randomly re-split the V2 dataset here.
#
# The generator already created train/validation/test and keeps benchmark.jsonl
# separate from training. Keeping those boundaries avoids accidental leakage
# from the generated corpus into evaluation.
# =============================================================================

print("Using generator-provided splits:")
print("Train      :", len(train_records))
print("Validation :", len(val_records))
print("Test       :", len(test_records))
print("Benchmark  :", len(benchmark_records))
print("✓ No additional random split performed.")


Using generator-provided splits:
Train      : 15624
Validation : 1953
Test       : 1954
Benchmark  : 2000
✓ No additional random split performed.


In [10]:
# %% [code]
# =============================================================================
# CELL 9 — SAVE A LOCAL COPY OF THE EXACT V2 SPLITS
# =============================================================================

def copy_file(src, dst):
    shutil.copy2(src, dst)
    print("✓", dst)

copy_file(TRAIN_PATH, PROCESSED_DIR / "train.jsonl")
copy_file(VAL_PATH, PROCESSED_DIR / "validation.jsonl")
copy_file(TEST_PATH, PROCESSED_DIR / "test.jsonl")
copy_file(BENCHMARK_PATH, PROCESSED_DIR / "benchmark.jsonl")


✓ /kaggle/working/data/processed/train.jsonl
✓ /kaggle/working/data/processed/validation.jsonl
✓ /kaggle/working/data/processed/test.jsonl
✓ /kaggle/working/data/processed/benchmark.jsonl


In [11]:
# %% [code]
# =============================================================================
# CELL 10 — DATASET INTEGRITY SUMMARY
# =============================================================================

assert len(train_records) > 0
assert len(val_records) > 0
assert len(test_records) > 0
assert len(benchmark_records) > 0

print("✓ All four SME-Ledger V2 JSONL files are present and non-empty.")
print("✓ Benchmark remains isolated from training.")


✓ All four SME-Ledger V2 JSONL files are present and non-empty.
✓ Benchmark remains isolated from training.


In [12]:
# %% [code]
# =============================================================================
# CELL 18 — TOKENIZER
# =============================================================================

print("=" * 80)
print("LOADING TOKENIZER")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN if "HF_TOKEN" in globals() else None,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD:", tokenizer.pad_token)
print("EOS:", tokenizer.eos_token)

LOADING TOKENIZER


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Vocab size: 262144
PAD: <pad>
EOS: <eos>


In [13]:
# %% [code]
# =============================================================================
# CELL 19 — SME-LEDGER V2 CHAT FORMATTING
# =============================================================================

def build_example(record):
    messages = record["messages"]

    # Prompt contains only the user turn. The generation prompt ends immediately
    # before the assistant response.
    prompt = tokenizer.apply_chat_template(
        [messages[0]],
        tokenize=False,
        add_generation_prompt=True,
    )

    target = messages[1]["content"]

    return {
        "prompt": prompt,
        "target": target,
    }

train_examples = [build_example(r) for r in train_records]
val_examples = [build_example(r) for r in val_records]
test_examples = [build_example(r) for r in test_records]

print("Train examples:", len(train_examples))
print("Val examples  :", len(val_examples))
print("Test examples :", len(test_examples))

print("\nPROMPT:")
print(train_examples[0]["prompt"][:1500])
print("\nTARGET:")
print(train_examples[0]["target"][:1000])


Train examples: 15624
Val examples  : 1953
Test examples : 1954

PROMPT:
<bos><start_of_turn>user
UIPB1SKBO1 Confirmed. You have received Ksh18,810.00 from Victor Kiptoo 0796617790 on 23/4/26 at 03:41. New M-PESA balance is Ksh108,890.68.<end_of_turn>
<start_of_turn>model


TARGET:
{"transaction_id":"UIPB1SKBO1","date":"2026-04-23","time":"03:41","type":"income","domain":"receive_money","entity":"Victor Kiptoo","amount":18810.0,"balance":108890.68,"fee":0.0,"reference":null}


In [14]:
# %% [code]
# =============================================================================
# CELL 20 — TOKENIZATION / LABEL-MASKED COLLATOR
# =============================================================================
#
# Only assistant tokens contribute to the loss.
# The model sees the complete user prompt, but we mask the prompt with -100.
# This is important for instruction tuning on the V2 messages.
# =============================================================================

def tokenize_example(example):
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
    )["input_ids"]

    target_ids = tokenizer(
        example["target"],
        add_special_tokens=False,
    )["input_ids"]

    target_ids = target_ids + [tokenizer.eos_token_id]

    if len(target_ids) >= MAX_LENGTH:
        target_ids = target_ids[:MAX_LENGTH - 1] + [tokenizer.eos_token_id]

    max_prompt_length = MAX_LENGTH - len(target_ids)

    if max_prompt_length <= 0:
        raise ValueError(
            "Target alone exceeds MAX_LENGTH. Increase MAX_LENGTH."
        )

    prompt_ids = prompt_ids[:max_prompt_length]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_dataset = Dataset.from_list([tokenize_example(x) for x in train_examples])
val_dataset = Dataset.from_list([tokenize_example(x) for x in val_examples])
test_dataset = Dataset.from_list([tokenize_example(x) for x in test_examples])

class CausalLMDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)

        input_ids = []
        attention_masks = []
        labels = []

        for feature in features:
            pad = max_len - len(feature["input_ids"])

            input_ids.append(
                feature["input_ids"] +
                [self.tokenizer.pad_token_id] * pad
            )
            attention_masks.append(
                feature["attention_mask"] +
                [0] * pad
            )
            labels.append(
                feature["labels"] +
                [-100] * pad
            )

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

collator = CausalLMDataCollator(tokenizer)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))


Train: 15624
Val  : 1953
Test : 1954


In [15]:
# %% [code]
# =============================================================================
# CELL 21 — LOAD BASE MODEL + NUMERICAL CHECK
# =============================================================================

cleanup_memory()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    token=HF_TOKEN if "HF_TOKEN" in globals() else None,
)

if torch.cuda.is_available():
    base_model = base_model.cuda()

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False

print("Model dtype :", next(base_model.parameters()).dtype)
print("Model device:", next(base_model.parameters()).device)
print_memory("BASE MODEL")

def check_model_parameters(model, name="model"):
    bad = []
    total = 0

    for param_name, param in model.named_parameters():
        total += param.numel()
        if not torch.isfinite(param).all():
            bad.append(param_name)

    print(f"{name}: checked {total:,} parameters; bad tensors={len(bad)}")

    if bad:
        print("First bad tensors:", bad[:20])
        raise RuntimeError(f"{name} contains NaN/Inf parameters.")

check_model_parameters(base_model, "BASE MODEL")

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Model dtype : torch.float32
Model device: cuda:0

MEMORY — BASE MODEL
CPU RSS: 1.927 GB
GPU allocated: 0.999 GB
GPU reserved : 1.0 GB
BASE MODEL: checked 268,098,176 parameters; bad tensors=0


In [16]:
# %% [code]
# =============================================================================
# CELL 22 — FORWARD TEST + LoRA
# =============================================================================
!pip install -q -U "torchao>=0.16.0"
def make_test_batch(dataset, batch_size=1):
    examples = [
        dataset[i]
        for i in range(min(batch_size, len(dataset)))
    ]
    batch = collator(examples)
    device = next(base_model.parameters()).device
    return {k: v.to(device) for k, v in batch.items()}

base_test_batch = make_test_batch(train_dataset)

base_model.eval()
with torch.no_grad():
    base_outputs = base_model(**base_test_batch)

assert torch.isfinite(base_outputs.logits).all()
assert torch.isfinite(base_outputs.loss)

print("Base forward loss:", base_outputs.loss.float().item())

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

check_model_parameters(model, "LoRA MODEL")

model.eval()
with torch.no_grad():
    lora_outputs = model(**base_test_batch)

assert torch.isfinite(lora_outputs.logits).all()
assert torch.isfinite(lora_outputs.loss)

print("LoRA forward loss:", lora_outputs.loss.float().item())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 56.7 MB/s eta 0:00:00


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Base forward loss: 2.3376851081848145
trainable params: 1,898,496 || all params: 269,996,672 || trainable%: 0.7032
LoRA MODEL: checked 269,996,672 parameters; bad tensors=0
LoRA forward loss: 2.3376851081848145


In [17]:
# %% [code]
# =============================================================================
# CELL 23 — GRADIENT SANITY CHECK
# =============================================================================

model.train()
model.zero_grad(set_to_none=True)

outputs = model(**base_test_batch)
loss = outputs.loss

assert torch.isfinite(loss)

loss.backward()

bad_gradients = []
total_gradient_norm_sq = 0.0

for name, parameter in model.named_parameters():
    if not parameter.requires_grad or parameter.grad is None:
        continue

    if not torch.isfinite(parameter.grad).all():
        bad_gradients.append(name)

    g = parameter.grad.detach().float().norm(2).item()
    total_gradient_norm_sq += g ** 2

gradient_norm = math.sqrt(total_gradient_norm_sq)

print("Loss          :", loss.float().item())
print("Gradient norm :", gradient_norm)
print("Bad gradients :", len(bad_gradients))

if bad_gradients or not math.isfinite(gradient_norm):
    raise RuntimeError("Gradient sanity check failed.")

model.zero_grad(set_to_none=True)
print("✓ Gradient test passed.")

Loss          : 2.3376851081848145
Gradient norm : 14.958776690949522
Bad gradients : 0
✓ Gradient test passed.


In [18]:
# %% [code]
# =============================================================================
# CELL 24 — TRAINING ARGUMENTS
# VERSION-ADAPTIVE / COMPATIBILITY SAFE
# =============================================================================

print("=" * 80)
print("BUILDING TRAINING ARGUMENTS")
print("=" * 80)

checkpoint_dir = MODELS_DIR / "checkpoints"

if checkpoint_dir.exists():
    shutil.rmtree(checkpoint_dir)

# -------------------------------------------------------------------------
# Inspect the installed TrainingArguments API
# -------------------------------------------------------------------------

import inspect

ta_params = inspect.signature(
    TrainingArguments.__init__
).parameters

print("Supported TrainingArguments parameters:")
print(", ".join(
    p for p in ta_params
    if p != "self"
))

# -------------------------------------------------------------------------
# Build arguments only when the installed Transformers version supports them
# -------------------------------------------------------------------------

training_kwargs = {
    "output_dir": str(checkpoint_dir),

    "num_train_epochs": NUM_EPOCHS,

    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,

    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,

    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,

    "max_grad_norm": MAX_GRAD_NORM,

    "fp16": False,
    "bf16": False,

    "logging_steps": 5,

    "save_total_limit": 2,

    "gradient_checkpointing": False,

    "report_to": "none",

    "remove_unused_columns": False,

    "seed": SEED,
    "data_seed": SEED,
}


# =========================================================================
# WARMUP
# =========================================================================

if "warmup_ratio" in ta_params:

    training_kwargs["warmup_ratio"] = WARMUP_RATIO

    print("✓ Using warmup_ratio")

elif "warmup_steps" in ta_params:

    # Calculate warmup steps manually.
    #
    # Number of optimizer steps per epoch:
    steps_per_epoch = math.ceil(
        len(train_dataset)
        / (
            TRAIN_BATCH_SIZE
            * GRADIENT_ACCUMULATION_STEPS
        )
    )

    total_steps = (
        steps_per_epoch
        * NUM_EPOCHS
    )

    warmup_steps = max(
        1,
        int(total_steps * WARMUP_RATIO)
    )

    training_kwargs["warmup_steps"] = warmup_steps

    print(
        f"✓ Using warmup_steps={warmup_steps} "
        f"(equivalent to ~{WARMUP_RATIO:.1%} warmup)"
    )

else:

    print("⚠ No warmup parameter supported.")


# =========================================================================
# EVALUATION STRATEGY
# =========================================================================

if "eval_strategy" in ta_params:

    training_kwargs["eval_strategy"] = "epoch"

    print("✓ Using eval_strategy")

elif "evaluation_strategy" in ta_params:

    training_kwargs["evaluation_strategy"] = "epoch"

    print("✓ Using evaluation_strategy")

else:

    print("⚠ Evaluation strategy unsupported.")


# =========================================================================
# SAVE STRATEGY
# =========================================================================

if "save_strategy" in ta_params:

    training_kwargs["save_strategy"] = "epoch"

    print("✓ Using save_strategy")

else:

    print("⚠ save_strategy unsupported.")


# =========================================================================
# BEST MODEL
# =========================================================================

if "load_best_model_at_end" in ta_params:

    training_kwargs["load_best_model_at_end"] = True

if "metric_for_best_model" in ta_params:

    training_kwargs["metric_for_best_model"] = "eval_loss"

if "greater_is_better" in ta_params:

    training_kwargs["greater_is_better"] = False


# =========================================================================
# LOGGING STRATEGY
# =========================================================================

if "logging_strategy" in ta_params:

    training_kwargs["logging_strategy"] = "steps"


# =========================================================================
# CREATE TRAINING ARGUMENTS
# =========================================================================

print("\nFinal TrainingArguments configuration:")
for key, value in training_kwargs.items():
    print(f"  {key:30s}: {value}")

training_args = TrainingArguments(
    **training_kwargs
)

print("\n✓ TrainingArguments created successfully.")


# =========================================================================
# CREATE TRAINER
# =========================================================================

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=collator,
)

print("✓ Trainer created successfully.")
print("=" * 80)

BUILDING TRAINING ARGUMENTS
Supported TrainingArguments parameters:
output_dir, per_device_train_batch_size, num_train_epochs, max_steps, learning_rate, lr_scheduler_type, lr_scheduler_kwargs, warmup_steps, optim, optim_args, weight_decay, adam_beta1, adam_beta2, adam_epsilon, optim_target_modules, gradient_accumulation_steps, average_tokens_across_devices, max_grad_norm, label_smoothing_factor, bf16, fp16, bf16_full_eval, fp16_full_eval, tf32, gradient_checkpointing, gradient_checkpointing_kwargs, torch_compile, torch_compile_backend, torch_compile_mode, use_liger_kernel, liger_kernel_config, use_cache, neftune_noise_alpha, torch_empty_cache_steps, auto_find_batch_size, logging_strategy, logging_steps, logging_first_step, log_on_each_node, logging_nan_inf_filter, include_num_input_tokens_seen, log_level, log_level_replica, disable_tqdm, report_to, run_name, project, trackio_space_id, trackio_bucket_id, trackio_static_space_id, eval_strategy, eval_steps, eval_delay, per_device_eval_bat

In [19]:
# %% [code]
# =============================================================================
# CELL 24 — TRAINING ARGUMENTS
# VERSION-ADAPTIVE / COMPATIBILITY SAFE
# =============================================================================

print("=" * 80)
print("BUILDING TRAINING ARGUMENTS")
print("=" * 80)

checkpoint_dir = MODELS_DIR / "checkpoints"

if checkpoint_dir.exists():
    shutil.rmtree(checkpoint_dir)

# -------------------------------------------------------------------------
# Inspect the installed TrainingArguments API
# -------------------------------------------------------------------------

import inspect

ta_params = inspect.signature(
    TrainingArguments.__init__
).parameters

print("Supported TrainingArguments parameters:")
print(", ".join(
    p for p in ta_params
    if p != "self"
))

# -------------------------------------------------------------------------
# Build arguments only when the installed Transformers version supports them
# -------------------------------------------------------------------------

training_kwargs = {
    "output_dir": str(checkpoint_dir),

    "num_train_epochs": NUM_EPOCHS,

    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,

    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,

    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,

    "max_grad_norm": MAX_GRAD_NORM,

    "fp16": False,
    "bf16": False,

    "logging_steps": 5,

    "save_total_limit": 2,

    "gradient_checkpointing": False,

    "report_to": "none",

    "remove_unused_columns": False,

    "seed": SEED,
    "data_seed": SEED,
}


# =========================================================================
# WARMUP
# =========================================================================

if "warmup_ratio" in ta_params:

    training_kwargs["warmup_ratio"] = WARMUP_RATIO

    print("✓ Using warmup_ratio")

elif "warmup_steps" in ta_params:

    # Calculate warmup steps manually.
    #
    # Number of optimizer steps per epoch:
    steps_per_epoch = math.ceil(
        len(train_dataset)
        / (
            TRAIN_BATCH_SIZE
            * GRADIENT_ACCUMULATION_STEPS
        )
    )

    total_steps = (
        steps_per_epoch
        * NUM_EPOCHS
    )

    warmup_steps = max(
        1,
        int(total_steps * WARMUP_RATIO)
    )

    training_kwargs["warmup_steps"] = warmup_steps

    print(
        f"✓ Using warmup_steps={warmup_steps} "
        f"(equivalent to ~{WARMUP_RATIO:.1%} warmup)"
    )

else:

    print("⚠ No warmup parameter supported.")


# =========================================================================
# EVALUATION STRATEGY
# =========================================================================

if "eval_strategy" in ta_params:

    training_kwargs["eval_strategy"] = "epoch"

    print("✓ Using eval_strategy")

elif "evaluation_strategy" in ta_params:

    training_kwargs["evaluation_strategy"] = "epoch"

    print("✓ Using evaluation_strategy")

else:

    print("⚠ Evaluation strategy unsupported.")


# =========================================================================
# SAVE STRATEGY
# =========================================================================

if "save_strategy" in ta_params:

    training_kwargs["save_strategy"] = "epoch"

    print("✓ Using save_strategy")

else:

    print("⚠ save_strategy unsupported.")


# =========================================================================
# BEST MODEL
# =========================================================================

if "load_best_model_at_end" in ta_params:

    training_kwargs["load_best_model_at_end"] = True

if "metric_for_best_model" in ta_params:

    training_kwargs["metric_for_best_model"] = "eval_loss"

if "greater_is_better" in ta_params:

    training_kwargs["greater_is_better"] = False


# =========================================================================
# LOGGING STRATEGY
# =========================================================================

if "logging_strategy" in ta_params:

    training_kwargs["logging_strategy"] = "steps"


# =========================================================================
# CREATE TRAINING ARGUMENTS
# =========================================================================

print("\nFinal TrainingArguments configuration:")
for key, value in training_kwargs.items():
    print(f"  {key:30s}: {value}")

training_args = TrainingArguments(
    **training_kwargs
)

print("\n✓ TrainingArguments created successfully.")


# =========================================================================
# CREATE TRAINER
# =========================================================================

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=collator,
)

print("✓ Trainer created successfully.")
print("=" * 80)

BUILDING TRAINING ARGUMENTS
Supported TrainingArguments parameters:
output_dir, per_device_train_batch_size, num_train_epochs, max_steps, learning_rate, lr_scheduler_type, lr_scheduler_kwargs, warmup_steps, optim, optim_args, weight_decay, adam_beta1, adam_beta2, adam_epsilon, optim_target_modules, gradient_accumulation_steps, average_tokens_across_devices, max_grad_norm, label_smoothing_factor, bf16, fp16, bf16_full_eval, fp16_full_eval, tf32, gradient_checkpointing, gradient_checkpointing_kwargs, torch_compile, torch_compile_backend, torch_compile_mode, use_liger_kernel, liger_kernel_config, use_cache, neftune_noise_alpha, torch_empty_cache_steps, auto_find_batch_size, logging_strategy, logging_steps, logging_first_step, log_on_each_node, logging_nan_inf_filter, include_num_input_tokens_seen, log_level, log_level_replica, disable_tqdm, report_to, run_name, project, trackio_space_id, trackio_bucket_id, trackio_static_space_id, eval_strategy, eval_steps, eval_delay, per_device_eval_bat

In [20]:
# %% [code]
# =============================================================================
# CELL 25 — FINAL PRE-TRAINING SANITY CHECK
# =============================================================================

trainer_model = trainer.model
first_parameter = next(trainer_model.parameters())

print("Model type :", type(trainer_model).__name__)
print("Device     :", first_parameter.device)
print("Dtype      :", first_parameter.dtype)

fresh_batch = collator([train_dataset[0]])
fresh_batch = {k: v.to(first_parameter.device) for k, v in fresh_batch.items()}

trainer_model.eval()
with torch.no_grad():
    outputs = trainer_model(**fresh_batch)

assert torch.isfinite(outputs.logits).all()
assert torch.isfinite(outputs.loss)

trainable = sum(p.numel() for p in trainer_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer_model.parameters())

print("Total params     :", f"{total:,}")
print("Trainable params :", f"{trainable:,}")
print("Trainable %      :", f"{100 * trainable / total:.4f}%")
print("Loss             :", outputs.loss.float().item())

trainer_model.train()
print("✓ ALL PRE-TRAINING CHECKS PASSED.")

Model type : PeftModelForCausalLM
Device     : cuda:0
Dtype      : torch.float32
Total params     : 269,996,672
Trainable params : 1,898,496
Trainable %      : 0.7032%
Loss             : 2.3376851081848145
✓ ALL PRE-TRAINING CHECKS PASSED.


In [21]:
# %% [code]
# =============================================================================
# CELL 26 — TRAIN
# =============================================================================

print("=" * 80)
print("STARTING LoRA TRAINING")
print("=" * 80)

print_memory("BEFORE TRAINING")

train_result = trainer.train()

print("=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)
print("Training loss:", train_result.training_loss)

print_memory("AFTER TRAINING")

STARTING LoRA TRAINING

MEMORY — BEFORE TRAINING
CPU RSS: 2.326 GB
GPU allocated: 1.514 GB
GPU reserved : 2.885 GB


Epoch,Training Loss,Validation Loss
1,0.058182,0.065612
2,0.055402,0.064278
3,0.051567,0.063674
4,0.071082,0.063350
5,0.055365,0.063166


TRAINING COMPLETE
Training loss: 0.0969480414351519

MEMORY — AFTER TRAINING
CPU RSS: 3.016 GB
GPU allocated: 1.536 GB
GPU reserved : 13.396 GB


In [22]:
# %% [code]
# =============================================================================
# CELL 27 — EVALUATE + SAVE LoRA
# =============================================================================

evaluation = trainer.evaluate()

print("=" * 80)
print("VALIDATION")
print("=" * 80)
print(json.dumps(evaluation, indent=2, default=str))

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

print("LoRA adapter saved:", LORA_DIR)

Training Loss,Validation Loss,Epoch
0.055365,0.063166,5


VALIDATION
{
  "eval_loss": 0.06316646188497543
}
LoRA adapter saved: /kaggle/working/models/lora


In [23]:
# %% [code]
# =============================================================================
# CELL 28 — MERGE LoRA
# =============================================================================

print("=" * 80)
print("MERGING LoRA INTO BASE MODEL")
print("=" * 80)

model.eval()
merged_model = model.merge_and_unload()

check_model_parameters(merged_model, "MERGED MODEL")

merged_model.config.use_cache = True

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model:", MERGED_DIR)

MERGING LoRA INTO BASE MODEL
MERGED MODEL: checked 268,098,176 parameters; bad tensors=0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model: /kaggle/working/models/merged


In [24]:
# %% [code]
# =============================================================================
# CELL 29 — GENERIC JSON EXTRACTION
# =============================================================================

def extract_json_value(text):
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    starts = [(i, ch) for i, ch in enumerate(text) if ch in "{["]

    for start, opening in starts:
        closing = "}" if opening == "{" else "]"
        depth = 0
        in_string = False
        escape = False

        for i in range(start, len(text)):
            ch = text[i]

            if in_string:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_string = False
                continue

            if ch == '"':
                in_string = True
            elif ch == opening:
                depth += 1
            elif ch == closing:
                depth -= 1
                if depth == 0:
                    candidate = text[start:i + 1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        break

    return None

def canonical_value(value):
    if isinstance(value, (dict, list)):
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
    return str(value).strip()

print("✓ Generic object/array JSON parser ready.")


✓ Generic object/array JSON parser ready.


In [25]:
# %% [code]
# =============================================================================
# CELL 30 — SINGLE V2 GENERATION TEST
# =============================================================================

def generate_from_user_message(model, user_content):
    messages = [{
        "role": "user",
        "content": user_content,
    }]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0, inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return {
        "raw_text": raw_text,
        "parsed": extract_json_value(raw_text),
    }

# Prefer a transaction example for the first qualitative test.
transaction_test = next(
    (
        r for r in test_records
        if isinstance(extract_json_value(r["messages"][1]["content"]), dict)
    ),
    test_records[0],
)

result = generate_from_user_message(
    merged_model,
    transaction_test["messages"][0]["content"],
)

print("=" * 80)
print("V2 GENERATION TEST")
print("=" * 80)
print("\nUSER:")
print(transaction_test["messages"][0]["content"])
print("\nREFERENCE:")
print(transaction_test["messages"][1]["content"])
print("\nMODEL RAW OUTPUT:")
print(result["raw_text"])
print("\nMODEL PARSED:")
print(json.dumps(result["parsed"], indent=2, ensure_ascii=False, default=str)
      if isinstance(result["parsed"], (dict, list))
      else result["parsed"])


V2 GENERATION TEST

USER:
UIORQKZRDA Confirmed. Ksh13,205.25 paid to NAIVAS SUPERMARKET via PayBill 400200. Account Number 123456789. Date 7/6/2026 at 08:34. New M-PESA balance is Ksh88,237.28.

REFERENCE:
{"transaction_id":"UIORQKZRDA","date":"2026-06-07","time":"08:34","type":"expense","domain":"paybill","entity":"NAIVAS SUPERMARKET","amount":13205.25,"balance":88237.28,"fee":0.0,"reference":"400200:123456789"}

MODEL RAW OUTPUT:
{"transaction_id":"UIORQKZRDA","date":"2026-06-07","time":"08:34","type":"expense","domain":"paybill","entity":"NAIVAS SUPERMARKET","amount":13205.25,"balance":88237.28,"fee":0.0,"reference":"400200:123456789"}

MODEL PARSED:
{
  "transaction_id": "UIORQKZRDA",
  "date": "2026-06-07",
  "time": "08:34",
  "type": "expense",
  "domain": "paybill",
  "entity": "NAIVAS SUPERMARKET",
  "amount": 13205.25,
  "balance": 88237.28,
  "fee": 0.0,
  "reference": "400200:123456789"
}


In [26]:
# %% [code]
# =============================================================================
# CELL 31 — QUANTITATIVE TEST-SET EVALUATION + LEDGER BUILD
# =============================================================================

EVAL_LIMIT = min(100, len(test_records))

evaluation_rows = []
parsed_transaction_records = []

for idx, record in enumerate(test_records[:EVAL_LIMIT]):
    user_content = record["messages"][0]["content"]
    reference_text = record["messages"][1]["content"]

    generated = generate_from_user_message(
        merged_model,
        user_content,
    )

    reference_value = extract_json_value(reference_text)
    generated_value = generated["parsed"]

    reference_is_json = isinstance(reference_value, (dict, list))

    if reference_is_json:
        exact_match = (
            canonical_value(reference_value)
            == canonical_value(generated_value)
        )
        valid_output = generated_value is not None
    else:
        exact_match = (
            generated["raw_text"].strip()
            == reference_text.strip()
        )
        valid_output = bool(generated["raw_text"].strip())

    evaluation_rows.append({
        "index": idx,
        "reference_is_json": reference_is_json,
        "valid_output": valid_output,
        "exact_match": exact_match,
    })

    # Build a rich transaction ledger from correctly shaped JSON objects.
    # We do not require exact match to inspect the model's extracted record.
    if isinstance(generated_value, dict):
        if "amount" in generated_value and (
            "domain" in generated_value or "type" in generated_value
        ):
            parsed_transaction_records.append(generated_value)

eval_df = pd.DataFrame(evaluation_rows)

evaluation = {
    "test_examples_evaluated": int(len(eval_df)),
    "valid_output_rate": float(eval_df["valid_output"].mean())
        if len(eval_df) else 0.0,
    "exact_match_rate": float(eval_df["exact_match"].mean())
        if len(eval_df) else 0.0,
    "json_examples": int(eval_df["reference_is_json"].sum())
        if len(eval_df) else 0,
}


# -------------------------------------------------------------------------
# Held-out unseen-template benchmark
# -------------------------------------------------------------------------

BENCHMARK_EVAL_LIMIT = min(50, len(benchmark_records))
benchmark_matches = []
benchmark_valid = []

for record in benchmark_records[:BENCHMARK_EVAL_LIMIT]:
    generated = generate_from_user_message(
        merged_model,
        record["messages"][0]["content"],
    )
    reference_value = extract_json_value(record["messages"][1]["content"])
    generated_value = generated["parsed"]

    if isinstance(reference_value, (dict, list)):
        benchmark_valid.append(generated_value is not None)
        benchmark_matches.append(
            canonical_value(reference_value)
            == canonical_value(generated_value)
        )
    else:
        benchmark_valid.append(bool(generated["raw_text"].strip()))
        benchmark_matches.append(
            generated["raw_text"].strip()
            == record["messages"][1]["content"].strip()
        )

evaluation["benchmark_examples_evaluated"] = BENCHMARK_EVAL_LIMIT
evaluation["benchmark_valid_output_rate"] = (
    float(np.mean(benchmark_valid)) if benchmark_valid else 0.0
)
evaluation["benchmark_exact_match_rate"] = (
    float(np.mean(benchmark_matches)) if benchmark_matches else 0.0
)

print("=" * 80)
print("V2 TEST EVALUATION")
print("=" * 80)
print(json.dumps(evaluation, indent=2))

# Rich ledger for deterministic analytics.
ledger_rows = []
for record in parsed_transaction_records:
    ledger_rows.append({
        "transaction_id": record.get("transaction_id"),
        "date": record.get("date"),
        "time": record.get("time"),
        "type": record.get("type"),
        "domain": record.get("domain"),
        "entity": record.get("entity"),
        "amount": record.get("amount"),
        "balance": record.get("balance"),
        "fee": record.get("fee"),
        "reference": record.get("reference"),
    })

ledger = pd.DataFrame(ledger_rows)

if not ledger.empty:
    ledger["amount"] = pd.to_numeric(ledger["amount"], errors="coerce")
    ledger["balance"] = pd.to_numeric(ledger["balance"], errors="coerce")
    ledger["fee"] = pd.to_numeric(ledger["fee"], errors="coerce")
    ledger["date"] = pd.to_datetime(ledger["date"], errors="coerce")

print("\nParsed transaction records:", len(ledger))
if not ledger.empty:
    display(ledger.head(10))


V2 TEST EVALUATION
{
  "test_examples_evaluated": 100,
  "valid_output_rate": 1.0,
  "exact_match_rate": 0.52,
  "json_examples": 100,
  "benchmark_examples_evaluated": 50,
  "benchmark_valid_output_rate": 1.0,
  "benchmark_exact_match_rate": 0.0
}

Parsed transaction records: 100


,transaction_id,date,time,type,domain,entity,amount,balance,fee,reference
0,UIORQKZRDA,2026-06-07,08:34,expense,paybill,NAIVAS SUPERMARKET,13205.25,88237.28,0.0,400200:123456789
1,UIWPH6C3Y1,2026-06-21,08:59,income,receive_money,Collins Otieno,13059.00,110298.06,0.0,None
2,UIM7E8416H,2026-01-01,03:01,expense,airtime,Safaricom Airtime,363.25,17364.10,0.0,None
3,UIWF33KZ1P,2026-07-31,12:28,income,receive_money,Mercy Odhiambo,24937.00,57145.34,0.0,None
4,UISBQU6UNY,2026-07-12,14:10,income,receive_money,Sarah Wambui,17755.00,58893.88,0.0,None
5,UI8DGWTPTQ,2026-02-20,11:49,expense,send_money,Peter Mwangi,17848.00,933.14,0.0,None
6,UIY5K0I7HP,2026-01-25,00:01,unknown,failed_transaction,MAMA MB0GA SHOP,1820.00,NaN,0.0,None
7,UI4AV5WJO6,2026-08-08,19:01,expense,airtime,Safaricom Airtime,74.00,4528.76,0.0,None
8,BNKNK651ZIP66,2026-04-04,02:54,expense,bank_transfer_sent,Alex Kimani,44869.95,23398.10,75.0,BNKNK651ZIP66
9,UIETP23SVC,2026-01-19,19:06,expense,fuliza_repayment,Fuliza M-PESA,6442.00,28903.94,0.0,None


In [27]:
# %% [code]
# =============================================================================
# CELL 32 — DETERMINISTIC FINANCIAL SUMMARY
# =============================================================================

def financial_summary(ledger):
    if ledger is None or ledger.empty:
        return {
            "transactions": 0,
            "income": 0.0,
            "expenses": 0.0,
            "net_cash_flow": 0.0,
            "latest_balance": None,
        }

    df = ledger.copy()
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["balance"] = pd.to_numeric(df["balance"], errors="coerce")

    income = df.loc[
        df["type"].astype(str).str.lower() == "income", "amount"
    ].sum()

    expenses = df.loc[
        df["type"].astype(str).str.lower() == "expense", "amount"
    ].sum()

    latest_balance = None
    valid = df.dropna(subset=["balance"])
    if not valid.empty:
        if "date" in valid.columns:
            valid = valid.sort_values(["date", "time"], na_position="last")
        latest_balance = float(valid["balance"].iloc[-1])

    return {
        "transactions": int(len(df)),
        "income": round(float(income), 2),
        "expenses": round(float(expenses), 2),
        "net_cash_flow": round(float(income - expenses), 2),
        "latest_balance": latest_balance,
    }

summary = financial_summary(ledger)

print("=" * 80)
print("FINANCIAL SUMMARY FROM MODEL-EXTRACTED V2 RECORDS")
print("=" * 80)
print(json.dumps(summary, indent=2))


FINANCIAL SUMMARY FROM MODEL-EXTRACTED V2 RECORDS
{
  "transactions": 100,
  "income": 654386.4,
  "expenses": 1018358.72,
  "net_cash_flow": -363972.32,
  "latest_balance": 48046.18
}


In [28]:
# %% [code]
# =============================================================================
# CELL 33 — FINAL FINANCIAL ARTIFACTS / REPORT
# =============================================================================

if not ledger.empty:
    ledger.to_csv(
        EXPORT_DIR / "transaction_ledger_v2.csv",
        index=False,
    )

eval_df.to_csv(
    EXPORT_DIR / "v2_test_evaluation.csv",
    index=False,
)

v2_report = {
    "project": "ADTC 2026 Gemma 3 270M + SME-Ledger V2",
    "base_model": BASE_MODEL,
    "dataset_root": str(V2_ROOT),
    "dataset": {
        "train": len(train_records),
        "validation": len(val_records),
        "test": len(test_records),
        "benchmark": len(benchmark_records),
    },
    "training": {
        "lora_rank": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "max_length": MAX_LENGTH,
        "dtype": str(MODEL_DTYPE),
    },
    "evaluation": evaluation,
    "financial_analytics": summary,
    "artifacts": {
        "lora_adapter": str(LORA_DIR),
        "merged_model": str(MERGED_DIR),
        "v2_test_evaluation": str(EXPORT_DIR / "v2_test_evaluation.csv"),
        "transaction_ledger": str(EXPORT_DIR / "transaction_ledger_v2.csv"),
    },
}

with open(
    REPORTS_DIR / "v2_model_report.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(v2_report, f, indent=2, ensure_ascii=False, default=str)

print("=" * 80)
print("V2 PIPELINE SUMMARY")
print("=" * 80)
print(json.dumps(v2_report, indent=2, ensure_ascii=False, default=str))


V2 PIPELINE SUMMARY
{
  "project": "ADTC 2026 Gemma 3 270M + SME-Ledger V2",
  "base_model": "google/gemma-3-270m-it",
  "dataset_root": "/kaggle/input/notebooks/wangapa106g/sme-ledger-data-genarator/sme_ledger_v2_dataset",
  "dataset": {
    "train": 15624,
    "validation": 1953,
    "test": 1954,
    "benchmark": 2000
  },
  "training": {
    "lora_rank": 8,
    "lora_alpha": 16,
    "learning_rate": 0.0001,
    "epochs": 5,
    "max_length": 768,
    "dtype": "torch.float32"
  },
  "evaluation": {
    "test_examples_evaluated": 100,
    "valid_output_rate": 1.0,
    "exact_match_rate": 0.52,
    "json_examples": 100,
    "benchmark_examples_evaluated": 50,
    "benchmark_valid_output_rate": 1.0,
    "benchmark_exact_match_rate": 0.0
  },
  "financial_analytics": {
    "transactions": 100,
    "income": 654386.4,
    "expenses": 1018358.72,
    "net_cash_flow": -363972.32,
    "latest_balance": 48046.18
  },
  "artifacts": {
    "lora_adapter": "/kaggle/working/models/lora",
    "me

In [29]:
# %% [code]
# =============================================================================
# CELL 34 — PREPARE LLAMA.CPP
# =============================================================================

LLAMA_CPP_DIR = PROJECT_ROOT / "llama.cpp"

if not LLAMA_CPP_DIR.exists():
    print("Cloning llama.cpp...")
    subprocess.run(
        [
            "git", "clone",
            "--depth", "1",
            "https://github.com/ggml-org/llama.cpp.git",
            str(LLAMA_CPP_DIR),
        ],
        check=True,
    )
else:
    print("llama.cpp already exists:", LLAMA_CPP_DIR)

print("llama.cpp directory ready.")

Cloning llama.cpp...


Cloning into '/kaggle/working/llama.cpp'...


llama.cpp directory ready.


In [30]:
# %% [code]
# =============================================================================
# CELL 35 — INSTALL CONVERTER DEPENDENCIES
# =============================================================================

requirements = LLAMA_CPP_DIR / "requirements" / "requirements-convert_hf_to_gguf.txt"

if requirements.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )
    print("✓ Converter requirements installed.")
else:
    print("Requirements file not found at:", requirements)
    print("Inspect the current llama.cpp checkout before continuing.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you

✓ Converter requirements installed.


In [31]:
# %% [code]
# =============================================================================
# CELL 36A — LOCATE GEMMA TOKENIZER
# =============================================================================

from pathlib import Path

print("=" * 80)
print("SEARCHING KAGGLE FILESYSTEM FOR GEMMA TOKENIZER")
print("=" * 80)

search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

matches = []

for root in search_roots:

    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for pattern in [
        "tokenizer.model",
        "tokenizer.json",
        "tokenizer_config.json",
    ]:

        found = list(
            root.rglob(pattern)
        )

        for path in found:

            matches.append(path)

            print(
                f"{pattern:25s} -> {path}"
            )


print("\n" + "=" * 80)
print("TOKENIZER.MODEL RESULTS")
print("=" * 80)

spm_files = [
    p for p in matches
    if p.name == "tokenizer.model"
]

if spm_files:

    print(
        f"✓ Found {len(spm_files)} tokenizer.model file(s)"
    )

    for p in spm_files:
        print(" ", p)

else:

    print(
        "❌ No tokenizer.model found anywhere under /kaggle/input or /kaggle/working"
    )


print("\n" + "=" * 80)
print("TOKENIZER.JSON RESULTS")
print("=" * 80)

json_files = [
    p for p in matches
    if p.name == "tokenizer.json"
]

for p in json_files:
    print(" ", p)

SEARCHING KAGGLE FILESYSTEM FOR GEMMA TOKENIZER

Searching: /kaggle/input

Searching: /kaggle/working
tokenizer.json            -> /kaggle/working/models/lora/tokenizer.json
tokenizer.json            -> /kaggle/working/models/merged/tokenizer.json
tokenizer.json            -> /kaggle/working/models/checkpoints/checkpoint-4885/tokenizer.json
tokenizer.json            -> /kaggle/working/models/checkpoints/checkpoint-3908/tokenizer.json
tokenizer_config.json     -> /kaggle/working/models/lora/tokenizer_config.json
tokenizer_config.json     -> /kaggle/working/models/merged/tokenizer_config.json
tokenizer_config.json     -> /kaggle/working/models/checkpoints/checkpoint-4885/tokenizer_config.json
tokenizer_config.json     -> /kaggle/working/models/checkpoints/checkpoint-3908/tokenizer_config.json

TOKENIZER.MODEL RESULTS
❌ No tokenizer.model found anywhere under /kaggle/input or /kaggle/working

TOKENIZER.JSON RESULTS
  /kaggle/working/models/lora/tokenizer.json
  /kaggle/working/models/merg

In [32]:
# %% [code]
# =============================================================================
# CELL 36 — GEMMA 3 → FP16 GGUF
# FIXED: USE ORIGINAL KAGGLE SENTENCEPIECE TOKENIZER
# =============================================================================

import shutil
import subprocess
import sys
import json
from pathlib import Path


print("=" * 80)
print("GEMMA 3 → FP16 GGUF CONVERSION")
print("=" * 80)


# =============================================================================
# 1. PATHS
# =============================================================================

MERGED_DIR = Path(MERGED_DIR)
GGUF_DIR = Path(GGUF_DIR)
LLAMA_CPP_DIR = Path(LLAMA_CPP_DIR)

# Original Gemma 3 tokenizer discovered in Kaggle
TOKENIZER_SOURCE = Path(
    "/kaggle/input/models/google/gemma-3/"
    "transformers/gemma-3-270m/2"
)

TOKENIZER_MODEL = (
    TOKENIZER_SOURCE /
    "tokenizer.model"
)

CONVERTER = (
    LLAMA_CPP_DIR /
    "convert_hf_to_gguf.py"
)

CONVERSION_DIR = (
    GGUF_DIR /
    "hf_gemma3_conversion"
)

HF_GGUF = (
    GGUF_DIR /
    "gemma3-financial-intelligence-f16.gguf"
)


# =============================================================================
# 2. VALIDATE PATHS
# =============================================================================

print("\nChecking paths...")

checks = {
    "Merged model": MERGED_DIR,
    "GGUF directory": GGUF_DIR,
    "llama.cpp converter": CONVERTER,
    "Tokenizer directory": TOKENIZER_SOURCE,
    "tokenizer.model": TOKENIZER_MODEL,
}


for name, path in checks.items():

    exists = path.exists()

    print(
        f"{name:25s}: "
        f"{'✓' if exists else '✗'} "
        f"{path}"
    )

    if not exists:

        raise FileNotFoundError(
            f"{name} does not exist:\n{path}"
        )


GGUF_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. INSPECT MERGED MODEL
# =============================================================================

print("\n" + "=" * 80)
print("MERGED MODEL")
print("=" * 80)

config_path = (
    MERGED_DIR /
    "config.json"
)

if not config_path.exists():

    raise FileNotFoundError(
        "Merged model is missing config.json"
    )


with open(
    config_path,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)


print(
    "Architecture:",
    config.get("architectures")
)

print(
    "Model type:",
    config.get("model_type")
)

print(
    "Vocab size:",
    config.get("vocab_size")
)

print(
    "Hidden size:",
    config.get("hidden_size")
)

print(
    "Layers:",
    config.get("num_hidden_layers")
)


# =============================================================================
# 4. CREATE CLEAN CONVERSION DIRECTORY
# =============================================================================

print("\n" + "=" * 80)
print("CREATING CLEAN CONVERSION DIRECTORY")
print("=" * 80)


if CONVERSION_DIR.exists():

    shutil.rmtree(
        CONVERSION_DIR
    )


CONVERSION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 5. COPY MODEL FILES
# =============================================================================

print("\nCopying trained model...")


# Copy every model/config file from merged directory
for source in MERGED_DIR.iterdir():

    if not source.is_file():
        continue

    # We deliberately replace tokenizer files below
    # with the original Gemma tokenizer.
    if source.name in {
        "tokenizer.json",
        "tokenizer_config.json",
        "tokenizer.model",
        "special_tokens_map.json",
        "added_tokens.json",
    }:
        continue

    destination = (
        CONVERSION_DIR /
        source.name
    )

    shutil.copy2(
        source,
        destination
    )

    print(
        f"✓ {source.name}"
    )


# =============================================================================
# 6. COPY ORIGINAL GEMMA TOKENIZER
# =============================================================================

print("\n" + "=" * 80)
print("INSTALLING ORIGINAL GEMMA TOKENIZER")
print("=" * 80)


# tokenizer.model
shutil.copy2(
    TOKENIZER_SOURCE / "tokenizer.model",
    CONVERSION_DIR / "tokenizer.model"
)

print(
    "✓ tokenizer.model"
)


# tokenizer.json
if (
    TOKENIZER_SOURCE /
    "tokenizer.json"
).exists():

    shutil.copy2(
        TOKENIZER_SOURCE / "tokenizer.json",
        CONVERSION_DIR / "tokenizer.json"
    )

    print(
        "✓ tokenizer.json"
    )


# tokenizer_config.json
if (
    TOKENIZER_SOURCE /
    "tokenizer_config.json"
).exists():

    shutil.copy2(
        TOKENIZER_SOURCE /
        "tokenizer_config.json",

        CONVERSION_DIR /
        "tokenizer_config.json"
    )

    print(
        "✓ tokenizer_config.json"
    )


# special tokens
if (
    TOKENIZER_SOURCE /
    "special_tokens_map.json"
).exists():

    shutil.copy2(
        TOKENIZER_SOURCE /
        "special_tokens_map.json",

        CONVERSION_DIR /
        "special_tokens_map.json"
    )

    print(
        "✓ special_tokens_map.json"
    )


# chat template
chat_template = (
    MERGED_DIR /
    "chat_template.jinja"
)

if chat_template.exists():

    shutil.copy2(
        chat_template,
        CONVERSION_DIR /
        "chat_template.jinja"
    )

    print(
        "✓ chat_template.jinja"
    )


# =============================================================================
# 7. VERIFY CONVERSION DIRECTORY
# =============================================================================

print("\n" + "=" * 80)
print("CONVERSION DIRECTORY")
print("=" * 80)


for file in sorted(
    CONVERSION_DIR.iterdir()
):

    if file.is_file():

        size_mb = (
            file.stat().st_size /
            (1024 ** 2)
        )

        print(
            f"{file.name:40s}"
            f"{size_mb:10.2f} MB"
        )


# =============================================================================
# 8. CRITICAL TOKENIZER CHECK
# =============================================================================

print("\n" + "=" * 80)
print("TOKENIZER CHECK")
print("=" * 80)


conversion_tokenizer = (
    CONVERSION_DIR /
    "tokenizer.model"
)


if not conversion_tokenizer.exists():

    raise RuntimeError(
        "CRITICAL: tokenizer.model was not copied."
    )


tokenizer_size = (
    conversion_tokenizer.stat().st_size
)


print(
    "✓ tokenizer.model present"
)

print(
    f"Tokenizer size: "
    f"{tokenizer_size / (1024 ** 2):.2f} MB"
)


# =============================================================================
# 9. REMOVE PREVIOUS GGUF
# =============================================================================

if HF_GGUF.exists():

    print(
        "\nRemoving previous GGUF..."
    )

    HF_GGUF.unlink()


# =============================================================================
# 10. BUILD CONVERSION COMMAND
# =============================================================================

cmd = [
    sys.executable,
    str(CONVERTER),
    str(CONVERSION_DIR),
    "--outfile",
    str(HF_GGUF),
    "--outtype",
    "f16",
]


print("\n" + "=" * 80)
print("CONVERSION COMMAND")
print("=" * 80)

print(
    " ".join(
        map(str, cmd)
    )
)


# =============================================================================
# 11. RUN CONVERSION
# =============================================================================

print("\n" + "=" * 80)
print("RUNNING LLAMA.CPP")
print("=" * 80)


result = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


# Print everything
print(
    result.stdout
)


# =============================================================================
# 12. CHECK CONVERSION
# =============================================================================

if result.returncode != 0:

    print("\n" + "=" * 80)
    print("❌ CONVERSION FAILED")
    print("=" * 80)

    raise RuntimeError(
        f"""
llama.cpp GGUF conversion failed.

Exit code:
{result.returncode}

The complete converter output is printed above.
"""
    )


# =============================================================================
# 13. VERIFY OUTPUT
# =============================================================================

if not HF_GGUF.exists():

    raise RuntimeError(
        """
llama.cpp exited successfully but the
GGUF file was not created.
"""
    )


gguf_size_gb = (
    HF_GGUF.stat().st_size /
    (1024 ** 3)
)


print("\n" + "=" * 80)
print("✓ FP16 GGUF CREATED")
print("=" * 80)

print(
    "Output:"
)

print(
    HF_GGUF
)

print(
    f"\nSize: {gguf_size_gb:.3f} GB"
)

GEMMA 3 → FP16 GGUF CONVERSION

Checking paths...
Merged model             : ✓ /kaggle/working/models/merged
GGUF directory           : ✓ /kaggle/working/models/gguf
llama.cpp converter      : ✓ /kaggle/working/llama.cpp/convert_hf_to_gguf.py
Tokenizer directory      : ✗ /kaggle/input/models/google/gemma-3/transformers/gemma-3-270m/2


FileNotFoundError: Tokenizer directory does not exist:
/kaggle/input/models/google/gemma-3/transformers/gemma-3-270m/2

In [ ]:
# %% [code]
# =============================================================================
# CELL 37 — BUILD LLAMA-QUANTIZE
# ROBUST KAGGLE VERSION
# =============================================================================

import subprocess
import shutil
from pathlib import Path


print("=" * 80)
print("BUILDING / LOCATING LLAMA-QUANTIZE")
print("=" * 80)


# =============================================================================
# 1. LLAMA.CPP PATHS
# =============================================================================

LLAMA_CPP_DIR = Path(LLAMA_CPP_DIR).resolve()

BUILD_DIR = (
    LLAMA_CPP_DIR /
    "build"
)

BIN_DIR = (
    BUILD_DIR /
    "bin"
)


print("llama.cpp source:")
print(LLAMA_CPP_DIR)

print("\nBuild directory:")
print(BUILD_DIR)

print("\nBinary directory:")
print(BIN_DIR)


# =============================================================================
# 2. VALIDATE LLAMA.CPP SOURCE
# =============================================================================

print("\n" + "=" * 80)
print("VALIDATING LLAMA.CPP SOURCE")
print("=" * 80)


if not LLAMA_CPP_DIR.exists():

    raise FileNotFoundError(
        f"""
llama.cpp directory does not exist:

{LLAMA_CPP_DIR}
"""
    )


CMAKE_FILE = (
    LLAMA_CPP_DIR /
    "CMakeLists.txt"
)


if not CMAKE_FILE.exists():

    raise FileNotFoundError(
        f"""
CMakeLists.txt was not found in:

{LLAMA_CPP_DIR}

Expected:

{CMAKE_FILE}
"""
    )


print(
    "✓ CMakeLists.txt found:"
)

print(
    CMAKE_FILE
)


# =============================================================================
# 3. SEARCH FOR EXISTING QUANTIZER
# =============================================================================

print("\n" + "=" * 80)
print("SEARCHING FOR EXISTING LLAMA-QUANTIZE")
print("=" * 80)


quantizer_candidates = [

    # Linux
    BUILD_DIR /
    "bin" /
    "llama-quantize",

    # Older llama.cpp layout
    BUILD_DIR /
    "bin" /
    "quantize",

    # Alternative build layout
    BUILD_DIR /
    "llama-quantize",

    # System PATH
    Path(
        shutil.which("llama-quantize")
        or "/__nonexistent__"
    ),

]


quantizer = None


for candidate in quantizer_candidates:

    if candidate.exists():

        quantizer = candidate.resolve()

        print(
            "✓ Found:",
            quantizer
        )

        break


# =============================================================================
# 4. CONFIGURE CMAKE
# =============================================================================

if quantizer is None:

    print("\nNo existing llama-quantize found.")

    print("\n" + "=" * 80)
    print("CONFIGURING CMAKE")
    print("=" * 80)


    BUILD_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


    configure_cmd = [

        "cmake",

        # Explicit source directory
        "-S",
        str(LLAMA_CPP_DIR),

        # Explicit build directory
        "-B",
        str(BUILD_DIR),

        # Kaggle-friendly build
        "-DGGML_NATIVE=OFF",

        # CPU build is sufficient for quantization
        "-DGGML_CUDA=OFF",

        # Build only what we need
        "-DLLAMA_BUILD_TOOLS=ON",

    ]


    print(
        "Running:"
    )

    print(
        " ".join(
            map(str, configure_cmd)
        )
    )


    configure_result = subprocess.run(
        configure_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )


    print(
        "\n" +
        configure_result.stdout
    )


    if configure_result.returncode != 0:

        raise RuntimeError(
            f"""
CMake configuration failed.

Exit code:
{configure_result.returncode}

The complete CMake output is printed above.
"""
        )


# =============================================================================
# 5. BUILD LLAMA-QUANTIZE
# =============================================================================

if quantizer is None:

    print("\n" + "=" * 80)
    print("BUILDING LLAMA-QUANTIZE")
    print("=" * 80)


    build_cmd = [

        "cmake",

        "--build",
        str(BUILD_DIR),

        "--config",
        "Release",

        "--target",
        "llama-quantize",

        "--",

        "-j2",

    ]


    print(
        "Running:"
    )

    print(
        " ".join(
            map(str, build_cmd)
        )
    )


    build_result = subprocess.run(
        build_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )


    print(
        "\n" +
        build_result.stdout
    )


    if build_result.returncode != 0:

        raise RuntimeError(
            f"""
llama-quantize build failed.

Exit code:
{build_result.returncode}

The complete build output is printed above.
"""
        )


# =============================================================================
# 6. FIND QUANTIZER AFTER BUILD
# =============================================================================

print("\n" + "=" * 80)
print("LOCATING QUANTIZER AFTER BUILD")
print("=" * 80)


quantizer_candidates = [

    BUILD_DIR /
    "bin" /
    "llama-quantize",

    BUILD_DIR /
    "bin" /
    "quantize",

    BUILD_DIR /
    "llama-quantize",

]


quantizer = None


for candidate in quantizer_candidates:

    if candidate.exists():

        quantizer = candidate.resolve()

        break


if quantizer is None:

    # Last-resort recursive search
    found = list(
        BUILD_DIR.rglob(
            "llama-quantize"
        )
    )

    if found:

        quantizer = found[0].resolve()


if quantizer is None:

    raise FileNotFoundError(
        f"""
llama-quantize was not found after building.

Searched:

{BUILD_DIR}
"""
    )


# =============================================================================
# 7. MAKE EXECUTABLE
# =============================================================================

try:

    quantizer.chmod(
        quantizer.stat().st_mode | 0o111
    )

except Exception as e:

    print(
        "Warning: could not change executable permissions:",
        e
    )


# =============================================================================
# 8. TEST QUANTIZER
# =============================================================================

print("\n" + "=" * 80)
print("TESTING LLAMA-QUANTIZE")
print("=" * 80)


test_result = subprocess.run(
    [
        str(quantizer),
        "--help",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


print(
    test_result.stdout[:5000]
)


if test_result.returncode not in [0, 1]:

    raise RuntimeError(
        f"""
llama-quantize executable could not be started.

Exit code:
{test_result.returncode}
"""
    )


# =============================================================================
# 9. FINAL RESULT
# =============================================================================

print("\n" + "=" * 80)
print("✓ LLAMA-QUANTIZE READY")
print("=" * 80)

print(
    "Quantizer:"
)

print(
    quantizer
)

print(
    "\nNext stage:"
)

print(
    "FP16 GGUF → Q4_K_M GGUF"
)

In [ ]:
# %% [code]
# =============================================================================
# CELL 38 — QUANTIZE TO Q4_K_M
# =============================================================================

Q4_GGUF = GGUF_DIR / "gemma3-financial-intelligence-Q4_K_M.gguf"

cmd = [
    str(quantizer),
    str(HF_GGUF),
    str(Q4_GGUF),
    "Q4_K_M",
]

print("Running:")
print(" ".join(map(str, cmd)))

subprocess.run(cmd, check=True)

print("✓ Quantized GGUF created:")
print(Q4_GGUF)

if Q4_GGUF.exists():
    print("Size GB:", round(Q4_GGUF.stat().st_size / 2**30, 3))

In [ ]:
# %% [code]
# =============================================================================
# CELL 39 — GGUF ARTIFACT CHECK
# =============================================================================

def file_info(path):
    path = Path(path)
    if not path.exists():
        return {"exists": False}
    return {
        "exists": True,
        "path": str(path),
        "size_mb": round(path.stat().st_size / 2**20, 2),
    }

gguf_report = {
    "merged_model": file_info(MERGED_DIR),
    "fp16_gguf": file_info(HF_GGUF),
    "q4_k_m_gguf": file_info(Q4_GGUF),
    "deployment_target": "llama.cpp-compatible GGUF",
    "quantization": "Q4_K_M",
}

with open(
    EXPORT_DIR / "gguf_export_report.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(gguf_report, f, indent=2)

print(json.dumps(gguf_report, indent=2))

In [ ]:
# %% [code]
# =============================================================================
# CELL 40 — FINAL MODEL CARD / REPRODUCIBILITY REPORT
# =============================================================================

final_report = {
    "project": "ADTC 2026 Gemma 3 270M Financial Intelligence — SME-Ledger V2",
    "base_model": BASE_MODEL,
    "dataset_root": str(V2_ROOT),
    "dataset": {
        "train": len(train_records),
        "validation": len(val_records),
        "test": len(test_records),
        "benchmark": len(benchmark_records),
    },
    "training": {
        "lora_rank": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "max_length": MAX_LENGTH,
        "dtype": str(MODEL_DTYPE),
    },
    "evaluation": evaluation,
    "artifacts": {
        "lora_adapter": str(LORA_DIR),
        "merged_model": str(MERGED_DIR),
        "fp16_gguf": str(HF_GGUF),
        "q4_k_m_gguf": str(Q4_GGUF),
    },
    "design": {
        "dataset": "SME-Ledger V2",
        "training_format": "generator-provided messages",
        "transaction_target_schema": [
            "transaction_id",
            "date",
            "time",
            "type",
            "domain",
            "entity",
            "amount",
            "balance",
            "fee",
            "reference",
        ],
        "benchmark_isolated": True,
        "financial_arithmetic": "deterministic Python/pandas",
    },
}

with open(
    REPORTS_DIR / "final_model_report.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False, default=str)

print("=" * 80)
print("PIPELINE COMPLETE")
print("=" * 80)

for key, value in final_report["artifacts"].items():
    print(f"{key:25s}: {value}")
